In [1]:
"""
==================================================================================
 CUSTOMER RETENTION INTELLIGENCE PLATFORM - SECONDARY DATASET (Olist)
 Phase 1: Data Understanding -> File 5 of 5: olist_orders_dataset.csv
==================================================================================
 Purpose : Explore the core orders table (used for Tableau dashboard).
           Contains order status and the full delivery timeline per order.
==================================================================================
"""

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 150)

DATE_COLS = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]


def section(title: str) -> None:
    print("\n" + "=" * 90)
    print(f" {title}")
    print("=" * 90)


# ----------------------------------------------------------------------------------
# 1. DATA COLLECTION
# ----------------------------------------------------------------------------------
section("1. DATA COLLECTION")

DATA_PATH = "/Users/meetnakrani/Library/Mobile Documents/com~apple~CloudDocs/Customer-Retention-intelligence-Platform/DataSet/RAW/secondary Data/olist_orders_dataset.csv"
df = pd.read_csv(DATA_PATH, parse_dates=DATE_COLS)

print(f"Dataset loaded successfully from: {DATA_PATH}")
print(f"Shape of dataset -> Rows: {df.shape[0]}, Columns: {df.shape[1]}")


# ----------------------------------------------------------------------------------
# 2. DATASET STRUCTURE
# ----------------------------------------------------------------------------------
section("2. DATASET STRUCTURE (First & Last Records)")
print("\n--- First 5 records ---")
print(df.head())
print("\n--- Last 5 records ---")
print(df.tail())

section("2.1 COLUMN NAMES & DATA TYPES")
print(df.dtypes)

section("2.2 MEMORY USAGE & GENERAL INFO")
df.info(memory_usage="deep")


# ----------------------------------------------------------------------------------
# 3. MISSING VALUE ANALYSIS
# ----------------------------------------------------------------------------------
section("3. MISSING VALUE ANALYSIS")

missing_count = df.isnull().sum()
missing_percent = (missing_count / len(df)) * 100
missing_summary = pd.DataFrame({
    "Missing Count": missing_count,
    "Missing %": missing_percent.round(2)
})
missing_summary = missing_summary[missing_summary["Missing Count"] > 0].sort_values("Missing %", ascending=False)

if missing_summary.empty:
    print("No missing values found in the dataset.")
else:
    print(missing_summary)
    print("\nNote: missing delivery/approval dates are expected for orders that are")
    print("cancelled, unavailable, or still in transit -> cross-check against order_status.")


# ----------------------------------------------------------------------------------
# 4. DUPLICATE RECORD CHECK
# ----------------------------------------------------------------------------------
section("4. DUPLICATE RECORD CHECK")
print(f"Fully duplicated rows      : {df.duplicated().sum()}")
print(f"Duplicated order_id count  : {df['order_id'].duplicated().sum()}")


# ----------------------------------------------------------------------------------
# 5. ORDER STATUS DISTRIBUTION
# ----------------------------------------------------------------------------------
section("5. ORDER STATUS DISTRIBUTION")
status_counts = df["order_status"].value_counts()
status_percent = df["order_status"].value_counts(normalize=True) * 100
status_summary = pd.DataFrame({"Count": status_counts, "Percentage": status_percent.round(2)})
print(status_summary)


# ----------------------------------------------------------------------------------
# 6. MISSING DATES CROSS-CHECKED AGAINST ORDER STATUS
# ----------------------------------------------------------------------------------
section("6. MISSING DELIVERY DATE BY ORDER STATUS")
undelivered = df[df["order_delivered_customer_date"].isnull()]
print(undelivered["order_status"].value_counts())


# ----------------------------------------------------------------------------------
# 7. DATE RANGE COVERAGE
# ----------------------------------------------------------------------------------
section("7. DATE RANGE COVERAGE")
for col in DATE_COLS:
    print(f"{col:35s} -> min: {df[col].min()}   max: {df[col].max()}")


# ----------------------------------------------------------------------------------
# 8. DELIVERY PERFORMANCE (Derived, for later Tableau KPIs)
# ----------------------------------------------------------------------------------
section("8. DELIVERY PERFORMANCE SNAPSHOT")

delivered = df.dropna(subset=["order_delivered_customer_date", "order_estimated_delivery_date"]).copy()
delivered["delivery_delay_days"] = (
    delivered["order_delivered_customer_date"] - delivered["order_estimated_delivery_date"]
).dt.days

on_time_pct = (delivered["delivery_delay_days"] <= 0).mean() * 100
print(f"Delivered orders analyzed        : {delivered.shape[0]}")
print(f"Delivered on or before estimate   : {round(on_time_pct, 2)}%")
print(f"Delivered late                    : {round(100 - on_time_pct, 2)}%")
print(f"Average delay (days, +ve = late)  : {round(delivered['delivery_delay_days'].mean(), 2)}")


# ----------------------------------------------------------------------------------
# 9. CARDINALITY CHECK
# ----------------------------------------------------------------------------------
section("9. CARDINALITY CHECK")
print(df.nunique().sort_values())


# ----------------------------------------------------------------------------------
# 10. INITIAL DATA QUALITY OBSERVATIONS
# ----------------------------------------------------------------------------------
section("10. INITIAL DATA QUALITY OBSERVATIONS")

observations = f"""
1. Dataset contains {df.shape[0]} unique orders spanning
   {df['order_purchase_timestamp'].min().date()} to {df['order_purchase_timestamp'].max().date()}.
2. 'order_id' is unique with no duplicates -> this is the central dimension
   table linking customers (customer_id), order_items, and payments.
3. {missing_summary.shape[0]} columns contain missing values, concentrated in
   'order_approved_at', 'order_delivered_carrier_date', and
   'order_delivered_customer_date' -> these are expected for orders with a
   non-'delivered' status (e.g. canceled, unavailable, shipped/in-transit)
   and should NOT be treated as data errors.
4. Order status is dominated by '{status_counts.index[0]}' at
   {status_percent.iloc[0].round(1)}% -> useful as a filter/segment in Tableau.
5. Delivery performance can be derived by comparing
   'order_delivered_customer_date' against 'order_estimated_delivery_date'
   -> a ready-made KPI for an on-time delivery rate chart.
6. Join keys: 'customer_id' -> olist_customers_dataset.csv,
   'order_id' -> olist_order_items_dataset.csv and olist_order_payments_dataset.csv.
"""
print(observations)

section("ORDERS DATASET - DATA UNDERSTANDING COMPLETE")


 1. DATA COLLECTION
Dataset loaded successfully from: /Users/meetnakrani/Library/Mobile Documents/com~apple~CloudDocs/Customer-Retention-intelligence-Platform/DataSet/RAW/secondary Data/olist_orders_dataset.csv
Shape of dataset -> Rows: 99441, Columns: 8

 2. DATASET STRUCTURE (First & Last Records)

--- First 5 records ---
                           order_id                       customer_id order_status order_purchase_timestamp   order_approved_at  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d    delivered      2017-10-02 10:56:33 2017-10-02 11:07:15   
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef    delivered      2018-07-24 20:41:37 2018-07-26 03:24:27   
2  47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089    delivered      2018-08-08 08:38:49 2018-08-08 08:55:23   
3  949d5b44dbf5de918fe9c16f97b45f8a  f88197465ea7920adcdbec7375364d82    delivered      2017-11-18 19:28:06 2017-11-18 19:45:59   
4  ad21c59c0840e6c